<a href="https://colab.research.google.com/github/toecm/iedi-mas/blob/main/CA_IEDI_0309.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- CA-IEDI with LAtency & Payload Measurement ---
# --- This addition was made to def automated_pipeline on March 9 2026 ---
# --- for the JCCI Conference Performance Evaluation ---
!pip install -q openai-whisper rapidfuzz pandas gradio datasets transformers torchaudio torch librosa pydub ffmpeg-python jiwer google-genai python-dotenv requests yt-dlp soundfile web3 eth-account

import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning, module="pydub")

import os
import glob
import torch
import whisper
import pandas as pd
import requests
import tempfile
import random
import shutil
import csv
import json
import re
import traceback
import threading
import concurrent.futures
import time
import hashlib
from pydub import AudioSegment
from pydub.generators import Sine
from rapidfuzz import process, fuzz
from google import genai
from google.genai import types
from dotenv import load_dotenv
from threading import Lock
from huggingface_hub import HfApi, hf_hub_download
from web3 import Web3
from eth_account import Account
import gradio as gr

# --- CONFIGURATION ---
HF_REPO_ID = "toecm/IEDID"

load_dotenv()

# Try Loading Keys (Colab or Local)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN') or os.getenv("HF_TOKEN")
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY') or os.getenv("GOOGLE_API_KEY")
    os.environ["PINATA_JWT"] = userdata.get('PINATA_JWT') or os.getenv("PINATA_JWT")
except (ImportError, Exception):
    pass

HF_TOKEN = os.getenv("HF_TOKEN")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
PINATA_JWT = os.getenv("PINATA_JWT")
PURECHAIN_RPC_URL = os.getenv("PURECHAIN_NETWORK_URL")
PURECHAIN_ID = int(os.getenv("PURECHAIN_ID", "0"))
PRIVATE_KEY = os.getenv("METAMASK_PRIVATE_KEY")

# --- DIRECTORY SETUP ---
DATASET_DIR = "/content/iuuy_datasets"
PROFILES_DIR = "/content/lab_profiles"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(PROFILES_DIR, exist_ok=True)

# --- HELPER: Warning Sound ---
def create_warning_beep():
    try:
        beep = Sine(1000).to_audio_segment(duration=500).apply_gain(5)
        path = os.path.join(tempfile.gettempdir(), "warning_beep.wav")
        beep.export(path, format="wav")
        return path
    except Exception as e:
        return None

WARNING_BEEP_PATH = create_warning_beep()

# --- OPTIMIZED MODEL MANAGER (From Script 2) ---
class GeminiManager:
    def __init__(self, api_key):
        self.api_key = api_key
        self.client = genai.Client(api_key=self.api_key) if self.api_key else None

        # UPGRADE: Standardization to 2.0 Flash for maximum speed/quality balance
        self.model_flash = "gemini-2.0-flash"
        self.last_used_model = "Idle"

        if self.client:
            print(f"🧠 Gemini Manager Connected: Standardized on {self.model_flash}")

    def generate_fast(self, prompt):
        if not self.client: raise Exception("Google API Key not found.")
        self.last_used_model = "Gemini 2.0 Flash"
        return self.client.models.generate_content(model=self.model_flash, contents=prompt)

    # Route "Smart" calls to Flash as well for speed, unless you specifically want Pro
    def generate_smart(self, prompt):
        return self.generate_fast(prompt)

    def get_status_string(self):
        return f"⚡ {self.last_used_model}"

gemini_manager = GeminiManager(GOOGLE_API_KEY) if GOOGLE_API_KEY else None

# --- HUGGING FACE MANAGER (From Script 1 - More Robust for UI) ---
class HFManager:
    def __init__(self):
        self.api = HfApi(token=HF_TOKEN)
        self.lock = Lock()

    def pull_datasets(self):
        print("⬇️ Pulling datasets & profiles...")
        try:
            files = self.api.list_repo_files(repo_id=HF_REPO_ID, repo_type="dataset")
            for f in files:
                target_dir = PROFILES_DIR if f.endswith(".json") else DATASET_DIR
                if f.endswith(".csv") or f.endswith(".json"):
                    hf_hub_download(repo_id=HF_REPO_ID, filename=f, repo_type="dataset", local_dir=target_dir, token=HF_TOKEN)
        except Exception as e:
            print(f"❌ HF Pull Error: {e}")
            self.seed_initial_data()

    def seed_initial_data(self):
        # Basic seed if HF fails
        p_path = os.path.join(PROFILES_DIR, "NSL Lab Trainer.json")
        if not os.path.exists(p_path):
            with open(p_path, 'w') as f: json.dump({"lab_name": "NSL", "jargon": {}, "pragmatic_rules": []}, f)

    def push_update(self, filepath, commit_msg="Update"):
        def _upload_task():
            filename = os.path.basename(filepath)
            try:
                self.api.upload_file(
                    path_or_fileobj=filepath,
                    path_in_repo=filename,
                    repo_id=HF_REPO_ID,
                    repo_type="dataset",
                    commit_message=commit_msg
                )
                print(f"✅ Background Sync Complete: {filename}")
            except Exception as e:
                print(f"❌ HF Push Error: {e}")
        threading.Thread(target=_upload_task, daemon=True).start()

    def upload_audio_sample(self, audio_path, dialect):
        clean_dialect = dialect.strip()
        filename = os.path.basename(audio_path)
        hf_path = f"audio/{clean_dialect}/{filename}"
        try:
            self.api.upload_file(path_or_fileobj=audio_path, path_in_repo=hf_path, repo_id=HF_REPO_ID, repo_type="dataset", commit_message=f"Add audio sample for {clean_dialect}")
            return hf_path
        except Exception as e:
            return None

hf_manager = HFManager()
hf_manager.pull_datasets()

# --- AGENT 1: INPUT (From Script 2 - Optimized Whisper) ---
class AgentInput:
    def __init__(self):
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"👂 Agent 1 (Input) Online: Loading Whisper Turbo on {device}...")
        try:
            # UPGRADE: Use Turbo for dialect speed
            self.model = whisper.load_model("turbo", device=device)
        except:
            print("⚠️ 'Turbo' not found (old whisper version). Fallback to 'medium'.")
            self.model = whisper.load_model("medium", device=device)

    def transcribe(self, audio_path, language="en"):
        if not audio_path: return []
        # UPGRADE: Context priming for dialects
        result = self.model.transcribe(audio_path, language=language, initial_prompt="Conversation in dialect. Borrow me your phone. How far?")
        return [{"speaker": "Speaker", "text": seg["text"].strip(), "start": seg["start"], "end": seg["end"]} for seg in result["segments"]]

# --- AGENT 2: INTERPRETATION (FIXED) ---
class AgentInterpretation:
    def __init__(self, gemini_manager_instance=None):
        self.df = pd.DataFrame()
        self.lookup_list = []
        self.gemini_manager = gemini_manager_instance
        self.lab_profile = self.load_profile_by_name("NSL Lab Trainer.json")


        # Persistent ThreadPool
        self.executor = concurrent.futures.ThreadPoolExecutor(max_workers=10)

        print("🧠 Agent 2 (Interpretation) Online: Persistent Pool Ready.")
        self.refresh_knowledge_base()

    # --- Feature: Profile Management ---
    def get_available_profiles(self):
        files = glob.glob(os.path.join(PROFILES_DIR, "*.json"))
        return [os.path.basename(f) for f in files]

    def load_profile_by_name(self, filename):
        path = os.path.join(PROFILES_DIR, filename)
        try:
            with open(path, 'r', encoding='utf-8') as f:
                self.lab_profile = json.load(f)
                return self.lab_profile
        except: return {}

    def save_specific_profile(self, filename, json_str):
        if not filename.endswith(".json"): filename += ".json"
        path = os.path.join(PROFILES_DIR, filename)
        try:
            with open(path, "w", encoding="utf-8") as f: json.dump(json.loads(json_str), f, indent=2)
            if 'hf_manager' in globals(): hf_manager.push_update(path, commit_msg=f"Update Profile: {filename}")
            return "✅ Saved"
        except Exception as e: return f"❌ Error: {e}"

    def get_current_profile_text(self):
        return json.dumps(self.lab_profile, indent=2)

    def refresh_knowledge_base(self):
        all_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
        df_list = []
        for filename in all_files:
            try:
                temp_df = pd.read_csv(filename, encoding='utf-8-sig', on_bad_lines='skip')
                temp_df["Dialect"] = os.path.basename(filename).replace(".csv", "")
                df_list.append(temp_df)
            except: pass
        if df_list:
            self.df = pd.concat(df_list, ignore_index=True)
            self.lookup_list = self.df["Utterance"].tolist()
        else:
            self.lookup_list = []

    # --- Helper: Normalize Keys ---
    def normalize_keys(self, data_list):
        """Ensures all keys are Capitalized to match UI expectations."""
        cleaned_list = []
        for item in data_list:
            new_item = {
                "Dialect": item.get("Dialect") or item.get("dialect", "Unknown"),
                "Clarification": item.get("Clarification") or item.get("clarification", "---"),
                "Tone": item.get("Tone") or item.get("tone", "---"),
                "Context": item.get("Context") or item.get("context", "---"),
                "Pragmatic Analysis": item.get("Pragmatic Analysis") or item.get("pragmatics", "---"),
                "Source": item.get("Source", "✨ AI Generated")
            }
            cleaned_list.append(new_item)
        return cleaned_list

    # --- Feature: AI Analysis ---
    def generate_unknown_analysis(self, text):
        if not self.gemini_manager: return []

        jargon_keys = list(self.lab_profile.get("jargon", {}).keys())

        # FIX: Prompt asks for Capitalized Keys
        prompt = f"""
        Analyze utterance: "{text}"
        Context/Jargon Keys: {jargon_keys}
        Task: Provide 3 distinct interpretations (Casual, Formal, or Cultural).
        Output Strictly JSON: [ {{ "Dialect": "General", "Clarification": "...", "Tone": "...", "Context": "...", "Pragmatics": "..." }} ]
        """
        try:
            response = self.gemini_manager.generate_fast(prompt)
            clean_text = re.sub(r"```json|```", "", response.text).strip()
            data = json.loads(clean_text)
            return self.normalize_keys(data) # Normalization Safety Net
        except:
            # FIX: Fallback uses Capitalized Keys
            return [{"Dialect": "Unknown", "Clarification": "Analysis Failed", "Tone": "---", "Context": "---", "Pragmatic Analysis": "Error"}]

    def adapt_with_ai(self, full_text, db_row):
        if not self.gemini_manager: return db_row["Clarification"], db_row["Pragmatic_Analysis"]
        prompt = f"""
        Ref Term: "{db_row['Utterance']}" = "{db_row['Clarification']}"
        User said: "{full_text}"
        Task: Adapt meaning to full sentence.
        Output JSON: {{ "clarification": "...", "pragmatics": "..." }}
        """
        try:
            response = self.gemini_manager.generate_fast(prompt)
            clean_json = re.search(r"\{.*\}", response.text, re.DOTALL)
            if clean_json:
                data = json.loads(clean_json.group(0))
                return data.get("clarification", db_row["Clarification"]), data.get("pragmatics", "AI Adapted Analysis")
        except: pass
        return db_row["Clarification"], db_row["Pragmatic_Analysis"]

    def detect_and_analyze(self, text, threshold=60): # Lowered threshold for better flexibility
        clean_text = text.lower().strip()
        seen_indices = set()

        # Lists to organize our "work to do"
        immediate_results = []
        partial_candidates = []

        # --- LAYER 1: LOCAL DB SCAN (Strict & Partial) ---
        if not self.df.empty:
            for index, row in self.df.iterrows():
                match_type = None

                # A. Regex Check
                try:
                    regex = str(row.get("Syntax_Pattern", ""))
                    if len(regex) > 2 and re.search(regex, clean_text, re.IGNORECASE):
                        match_type = "Regex_Match"
                except: pass

                # B. Substring Check
                if not match_type:
                    db_str = str(row["Utterance"]).strip().lower()
                    if len(db_str) > 3 and db_str in clean_text:
                        match_type = "Exact_Substring"

                if match_type:
                    seen_indices.add(index)
                    immediate_results.append({
                        "Source": f"💎 Database ({match_type})", "Dialect": row["Dialect"],
                        "Clarification": row["Clarification"], "Tone": row.get("Tone_Category", "---"),
                        "Context": row.get("Linguistic_Context", "---"), "Pragmatic Analysis": row.get("Pragmatic_Analysis", "---")
                    })

        # --- LAYER 2: FUZZY MATCH (Restored!) ---
        # This catches "Borrow me your phone" when DB has "Borrow me your pen"
        if not self.lookup_list: self.lookup_list = []
        # specific scorer 'token_set_ratio' handles reordered/replaced words better
        matches = process.extract(clean_text, self.lookup_list, scorer=fuzz.token_set_ratio, limit=5)

        for match_str, score, index in matches:
            if score >= threshold and index not in seen_indices and index < len(self.df):
                seen_indices.add(index)
                row = self.df.iloc[index]
                # We treat high-score fuzzy matches as "Candidates" for AI Adaptation
                # because "pen" vs "phone" requires changing the explanation slightly.
                partial_candidates.append({
                    "row": row,
                    "match_len": score, # Use score as weight
                    "type": f"Fuzzy ({score}%)"
                })

        # --- LAYER 3: PROFILE JARGON CHECK (Restored!) ---
        # Checks 'Nigerian Persona.json' specifically for the word "Borrow"
        jargon_dict = self.lab_profile.get("jargon", {})
        for term, definition in jargon_dict.items():
            # Check if the Jargon term exists in the text
            if term.lower() in clean_text:
                immediate_results.append({
                    "Source": f"📜 Profile Rule ({term})",
                    "Dialect": self.lab_profile.get("lab_name", "Profile"),
                    "Clarification": definition,
                    "Tone": "Detected Jargon",
                    "Context": f"Found in {self.lab_profile.get('lab_name')} Profile",
                    "Pragmatic Analysis": "Direct Profile Match"
                })

        # --- LAYER 4: EXECUTE AI (With Candidates) ---
        final_results = list(immediate_results)

        # Sort candidates by score
        partial_candidates.sort(key=lambda x: x["match_len"], reverse=True)
        top_candidates = partial_candidates[:3]

        # A. Launch Fallback (The "General" AI guess)
        fallback_future = self.executor.submit(self.generate_unknown_analysis, text)

        # B. Launch Adapters (Fixing "Pen" -> "Phone")
        db_futures = {}
        for cand in top_candidates:
            f = self.executor.submit(self.adapt_with_ai, text, cand["row"])
            db_futures[f] = cand

        # C. Wait
        done, not_done = concurrent.futures.wait(
            list(db_futures.keys()) + [fallback_future],
            timeout=4.5,
            return_when=concurrent.futures.ALL_COMPLETED
        )

        # Collect DB Results
        for f in db_futures:
            if f in done:
                try:
                    clar, prag = f.result()
                    cand = db_futures[f]
                    final_results.append({
                        "Source": f"💎 DB + AI ({cand['type']})", "Dialect": cand["row"]["Dialect"],
                        "Clarification": clar, "Tone": cand["row"].get("Tone_Category", "---"),
                        "Context": cand["row"].get("Linguistic_Context", "---"), "Pragmatic Analysis": prag
                    })
                except: pass

        # Collect Fallback if needed (fill up to 3)
        if len(final_results) < 3 and fallback_future in done:
            try:
                # Helper normalize check just in case
                res = self.normalize_keys(fallback_future.result())
                final_results += res
            except: pass

        if not final_results:
             final_results.append({"Source": "⚠️ AI Timeout", "Dialect": "---", "Clarification": "System Busy", "Tone": "---", "Context": "---", "Pragmatic Analysis": "---"})

        return final_results[:3]

    # --- Helpers ---
    def get_rich_suggestions(self, text, dialect):
        if not self.gemini_manager: return []
        prompt = f"""interpret "{text}" ({dialect}). Output 3 JSON options: [{{ "clarification": "", "tone": "", "context": "", "pragmatics": "" }}]"""
        try:
            res = self.gemini_manager.generate_fast(prompt)
            return json.loads(re.sub(r"```json|```", "", res.text).strip())
        except: return []

    def generate_syntax_pattern(self, utterance):
        safe = r"\b" + re.escape(utterance.lower()) + r"\b"
        if not self.gemini_manager: return safe
        try:
            pat = self.gemini_manager.generate_fast(f"Regex for: '{utterance}'. Return ONLY regex string.").text.strip().replace("`", "")
            re.compile(pat)
            return pat
        except: return safe

# --- AGENT 4: TRUST (From Script 1 - Detailed Tracking) ---
class AgentTrust:
    def __init__(self):
        self.lock = Lock()
        self.active_tasks = 0
        print("🛡️ Agent 4 (Trust) Online: Async Saving & Tracking Enabled.")

        # --- BLOCKCHAIN SETUP ---
        self.w3 = None
        self.account = None
        self.chain_id = PURECHAIN_ID

        if PURECHAIN_RPC_URL and PRIVATE_KEY:
            try:
                self.w3 = Web3(Web3.HTTPProvider(PURECHAIN_RPC_URL))
                self.account = Account.from_key(PRIVATE_KEY)
                print(f"🛡️ Agent 4 (Trust) Online: 🔗 PureChain Connected ({self.account.address[:6]}...)")
            except Exception as e:
                print(f"⚠️ Blockchain Connection Failed: {e}")

    def log_to_ipfs(self, data):
        if not PINATA_JWT: return "Local-Log-Only"
        headers = {"Authorization": f"Bearer {PINATA_JWT}"}
        try:
            res = requests.post("https://api.pinata.cloud/pinning/pinJSONToIPFS", headers=headers, json=data)
            return res.json().get("IpfsHash", "Error")
        except: return "IPFS_Fail"

    def stamp_on_chain(self, data_dict):
        if not self.w3 or not self.account: return "Skipped (No Chain)"
        try:
            data_str = json.dumps(data_dict, sort_keys=True)
            data_hash = hashlib.sha256(data_str.encode("utf-8")).hexdigest()
            tx = {
                'nonce': self.w3.eth.get_transaction_count(self.account.address),
                'to': self.account.address,
                'value': 0,
                'gas': 25000,
                'gasPrice': 0,
                'chainId': self.chain_id,
                'data': "0x" + data_hash
            }
            signed_tx = self.w3.eth.account.sign_transaction(tx, PRIVATE_KEY)
            tx_hash = self.w3.eth.send_raw_transaction(signed_tx.rawTransaction)
            return self.w3.to_hex(tx_hash)
        except Exception as e:
            return "Tx_Failed"

    def check_if_exists(self, utterance, dialect, brain_agent, clarification="", tone=""):
        if brain_agent.df.empty: return False
        clean_text = utterance.strip().lower()
        clean_dialect = dialect.strip()
        match = brain_agent.df[
            (brain_agent.df["Utterance"].str.strip().str.lower() == clean_text) &
            (brain_agent.df["Dialect"].str.strip() == clean_dialect) &
            (brain_agent.df["Clarification"].str.strip() == clarification.strip())
        ]
        return not match.empty

    def process_batch_feedback(self, dataframes, brain_agent, audio_path=None):
        all_rows = []
        for df in dataframes:
            if df is not None and not df.empty:
                for _, row in df.iterrows(): all_rows.append(row)

        def _batch_task():
            self.active_tasks += 1
            try:
                local_added = 0
                for row in all_rows:
                    if row["Source"] == "---" or not row["Utterance"]: continue
                    exists = self.check_if_exists(row["Utterance"], row["Dialect"], brain_agent, row["Clarification"], row["Tone"])
                    if not exists:
                        syntax = r"\b" + re.escape(row["Utterance"].lower()) + r"\b"
                        self.update_dataset_csv(row["Dialect"], row["Utterance"], row["Clarification"], row["Tone"], row["Context"], syntax, audio_path, row["Pragmatic Analysis"])

                        # Provenance
                        payload = {"action": "Batch_Add", "utterance": row["Utterance"], "timestamp": pd.Timestamp.now().isoformat()}
                        self.log_to_ipfs(payload)
                        self.stamp_on_chain(payload)
                        local_added += 1
                if local_added > 0: brain_agent.refresh_knowledge_base()
            finally:
                self.active_tasks -= 1

        threading.Thread(target=_batch_task, daemon=True).start()
        return f"✅ Batch Process Started! (Syncing to PureChain...)"

    def process_feedback(self, action, original_text, dialect, clarification, tone, context, brain_agent, audio_path=None, pragmatics=""):
        timestamp = pd.Timestamp.now().isoformat()
        feedback_data = {
            "original": original_text, "dialect": dialect, "clarification": clarification,
            "tone": tone, "linguistic_context": context, "pragmatics": pragmatics, "action": action, "timestamp": timestamp
        }

        def _background_save_task():
            self.active_tasks += 1
            try:
                print(f"⏳ Background Save Started for: {original_text[:20]}...")
                cid = self.log_to_ipfs(feedback_data)
                chain_payload = {"ipfs_cid": cid, "data_hash": feedback_data}
                tx_hash = self.stamp_on_chain(chain_payload)

                if action in ["Suggest Update", "Accept", "Force Overwrite"]:
                    syntax = r"\b" + re.escape(original_text.lower()) + r"\b"
                    if brain_agent:
                        try: syntax = brain_agent.generate_syntax_pattern(original_text)
                        except: pass

                    self.update_dataset_csv(dialect, original_text, clarification, tone, context, syntax, audio_path, pragmatics)
                    if brain_agent: brain_agent.refresh_knowledge_base()
                    print(f"✅ Save Complete.\n   📍 IPFS: {cid}\n   🔗 PureChain TX: {tx_hash}")
            except Exception as e:
                print(f"❌ Background Save Failed: {e}")
            finally:
                self.active_tasks -= 1

        threading.Thread(target=_background_save_task, daemon=True).start()
        return f"✅ Request Queued! (Stamping on PureChain...)"

    def update_dataset_csv(self, dialect, utterance, clarification, tone, context, syntax, audio_path=None, pragmatics=""):
        clean_dialect = dialect.strip().title()
        if not clean_dialect.endswith("English") and not clean_dialect.endswith("Dialect"): clean_dialect += " Dialect"
        filepath = os.path.join(DATASET_DIR, f"{clean_dialect}.csv")

        with self.lock:
            if not os.path.exists(filepath):
                new_df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "Pragmatic_Analysis", "file_name"])
                new_df.to_csv(filepath, index=False)

            try: df = pd.read_csv(filepath, encoding='utf-8-sig', on_bad_lines='skip')
            except: df = pd.DataFrame(columns=["Utterance", "Dialect", "Clarification", "Tone_Category", "Linguistic_Context", "Syntax_Pattern", "Pragmatic_Analysis", "file_name"])

            for col in ["Tone_Category", "Linguistic_Context", "file_name", "Syntax_Pattern", "Pragmatic_Analysis", "Clarification"]:
                if col not in df.columns: df[col] = "---"

            final_audio = ""
            if audio_path and os.path.exists(audio_path):
                ext = os.path.splitext(audio_path)[1]
                unique_name = f"{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}_{random.randint(1000,9999)}{ext}"
                new_path = os.path.join(os.path.dirname(audio_path), unique_name)
                try:
                    shutil.copy2(audio_path, new_path)
                    final_audio = hf_manager.upload_audio_sample(new_path, dialect)
                except: final_audio = "Error_Saving_Audio"

            new_row = pd.DataFrame([{
                "Utterance": utterance, "Dialect": clean_dialect, "Clarification": clarification,
                "Tone_Category": tone, "Linguistic_Context": context,
                "Pragmatic_Analysis": pragmatics,
                "Syntax_Pattern": syntax, "file_name": final_audio
            }])

            final_df = pd.concat([df, new_row], ignore_index=True)
            final_df.to_csv(filepath, index=False, quoting=csv.QUOTE_ALL)
            hf_manager.push_update(filepath, f"Update: {utterance}")
            return "Saved"

# --- AGENT 3: UX (From Script 1 - Full UI) ---
class AgentUX:
    def __init__(self, input_agent, brain_agent, trust_agent):
        self.input = input_agent
        self.brain = brain_agent
        self.trust = trust_agent
        self.last_audio_path = None
        self.suggestion_cache = {}
        print("🎨 Agent 3 (UX) Online: Interface Ready.")

    def get_quota_status(self):
        if self.brain.gemini_manager: return self.brain.gemini_manager.get_status_string()
        return "Manager not active"

    def check_background_status(self):
        tasks = self.trust.active_tasks
        if tasks > 0:
            return f"🔄 Processing {tasks} background task(s)..."
        return "✅ System Idle (Ready)"

    def automated_pipeline(self, audio_path, language="en"):
        import time
        import sys

        if not audio_path:
            empty = pd.DataFrame(columns=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"])
            return empty, empty, empty, "Waiting for Input...", self.get_quota_status()

        self.last_audio_path = audio_path

        # ==========================================
        # 1. Measure Edge Transcription (Whisper)
        # ==========================================
        start_edge = time.time()
        segments = self.input.transcribe(audio_path, language)
        edge_latency = time.time() - start_edge

        list_1, list_2, list_3 = [], [], []

        # Trackers for cloud metrics
        cloud_latencies = []
        total_payload_size = 0

        for seg in segments:
            raw = seg["text"]

            # ==========================================
            # 2. Measure Payload Size
            # ==========================================
            payload = {"text": raw, "persona": self.brain.lab_profile.get("lab_name", "Unknown")}
            total_payload_size += sys.getsizeof(str(payload))

            # ==========================================
            # 3. Measure Cloud LLM Inference (Gemini via DMM)
            # ==========================================
            start_cloud = time.time()
            possible_interpretations = self.brain.detect_and_analyze(raw)
            cloud_latency = time.time() - start_cloud
            cloud_latencies.append(cloud_latency)

            # Organize DataFrames
            def get_interp(idx):
                if idx < len(possible_interpretations): return possible_interpretations[idx]
                return {"Source": "---", "Dialect": "---", "Clarification": "---", "Tone": "---", "Context": "---", "Pragmatic Analysis": "---"}

            def make_row(interp):
                return {
                    "Source": interp["Source"], "Speaker": seg["speaker"], "Utterance": raw,
                    "Dialect": interp["Dialect"], "Clarification": interp["Clarification"],
                    "Tone": interp["Tone"], "Context": interp["Context"],
                    "Pragmatic Analysis": interp.get("Pragmatic Analysis", "---")
                }

            list_1.append(make_row(get_interp(0)))
            list_2.append(make_row(get_interp(1)))
            list_3.append(make_row(get_interp(2)))

        # ==========================================
        # 4. Calculate Totals & Format Output
        # ==========================================
        total_cloud_latency = sum(cloud_latencies)
        total_latency = edge_latency + total_cloud_latency

        # This will print the metrics directly into the "Analysis Log" Textbox in the UI
        metrics_log = (
            f"✅ Analysis Complete\n\n"
            f"📊 PERFORMANCE METRICS:\n"
            f"⏱️ Edge Latency (Whisper): {edge_latency:.3f} seconds\n"
            f"📦 Upstream Payload Size: {total_payload_size} bytes\n"
            f"☁️ Cloud Latency (Gemini): {total_cloud_latency:.3f} seconds\n"
            f"🚀 Total Round-Trip Time: {total_latency:.3f} seconds"
        )

        return pd.DataFrame(list_1), pd.DataFrame(list_2), pd.DataFrame(list_3), metrics_log, self.get_quota_status()

    def launch(self):
        existing_dialects = []
        if os.path.exists(DATASET_DIR):
            csv_files = glob.glob(os.path.join(DATASET_DIR, "*.csv"))
            existing_dialects = [os.path.basename(f).replace(".csv", "") for f in csv_files]
        dropdown_choices = existing_dialects + ["+ Add New Dialect"]
        available_profiles = self.brain.get_available_profiles()

        custom_css = """
        #red_btn { background-color: #FF0000 !important; color: white !important; font-weight: bold; }
        """

        with gr.Blocks(theme=gr.themes.Soft(), css=custom_css) as ui:
            gr.Markdown("## 🌍 CA-IEDI: Active Listening & English Language Mediator (Turbo Edition)")
            warning_player = gr.Audio(visible=False, autoplay=True)
            status_timer = gr.Timer(value=1.0)

            with gr.Tabs():
                with gr.Tab("🎙️ Live Analysis"):
                    with gr.Row():
                        with gr.Column(scale=1):
                            audio_input = gr.Audio(label="Step 1: Speak/Upload", sources=["microphone", "upload"], type="filepath")
                            lang_select = gr.Dropdown(["en", "ko", "fr"], value="en", label="Step 2: Language (Optional)")
                            analyze_btn = gr.Button("Re-Run Analysis 🔄", variant="secondary")

                            # STATUS BOXES
                            quota_display = gr.Textbox(label="📊 Model Status", value=self.get_quota_status(), interactive=False)
                            background_status_display = gr.Textbox(label="⚙️ System Status", value="Checking...", interactive=False)

                        with gr.Column(scale=3):
                            status_box = gr.Textbox(label="Analysis Log", interactive=False)
                            with gr.Row():
                                with gr.Column():
                                    gr.Markdown("### 🥇 Result 1")
                                    results_1 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 1", type="pandas", wrap=True)
                                with gr.Column():
                                    gr.Markdown("### 🥈 Result 2")
                                    results_2 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 2", type="pandas", wrap=True)
                                with gr.Column():
                                    gr.Markdown("### 🥉 Result 3")
                                    results_3 = gr.Dataframe(headers=["Source", "Speaker", "Utterance", "Dialect", "Clarification", "Tone", "Context", "Pragmatic Analysis"], interactive=False, label="Option 3", type="pandas", wrap=True)

                    gr.Markdown("### ✍️ Active Feedback Loop")
                    with gr.Row():
                        with gr.Column(scale=1):
                            orig_text_state = gr.Textbox(visible=True, label="Original Text (For Manual Edits)")
                            with gr.Row():
                                dialect_dropdown = gr.Dropdown(choices=dropdown_choices, label="Select Dialect", interactive=True)
                                new_dialect_input = gr.Textbox(label="Enter New Dialect Name", visible=False, interactive=True)
                        with gr.Column(scale=1):
                            suggestion_dropdown = gr.Dropdown(label="Suggest Clarification", choices=[], allow_custom_value=True, interactive=True)
                            selected_tone_state = gr.Textbox(label="Linguistic Tone", interactive=True)
                            selected_context_state = gr.TextArea(label="Linguistic Context", interactive=True, lines=2)
                            selected_pragmatics_state = gr.TextArea(label="Pragmatic Analysis", interactive=True, lines=2)

                    with gr.Row():
                        btn_accept = gr.Button("✅ Batch Accept / Verify All Results", variant="secondary")
                        btn_suggest = gr.Button("💾 Suggest Specific Edit", variant="primary")
                        btn_overwrite = gr.Button("⚠️ Confirm Overwrite", variant="stop", visible=False, elem_id="red_btn")

                    feedback_out = gr.Markdown()

                with gr.Tab("⚙️ Lab Context"):
                    gr.Markdown("### Profile Manager")
                    with gr.Row():
                        profile_selector = gr.Dropdown(choices=available_profiles, value="NSL Lab Trainer.json", label="Select Profile")
                        profile_filename = gr.Textbox(label="Filename (Edit to create new)", value="NSL Lab Trainer.json")
                    profile_editor = gr.Code(value=self.brain.get_current_profile_text(), language="json", label="Profile Content", lines=20)
                    save_profile_btn = gr.Button("💾 Save Profile", variant="primary")
                    profile_status = gr.Textbox(label="System Response", interactive=False)

            # --- EVENT LOGIC ---
            status_timer.tick(self.check_background_status, outputs=[background_status_display])

            def update_suggestions_rich(text, dialect):
                try:
                    if not text or not dialect or dialect == "+ Add New Dialect":
                        return gr.update(choices=[]), "", "", "", self.get_quota_status()
                    suggestions_data = self.brain.get_rich_suggestions(text, dialect)
                    self.suggestion_cache = {}
                    display_choices = []
                    if not suggestions_data: return gr.update(choices=["No suggestions"]), "", "", "", self.get_quota_status()
                    for item in suggestions_data:
                        clar, tone, ctx = item.get("clarification", ""), item.get("tone", ""), item.get("context", "")
                        prag = item.get("pragmatics", "Auto-generated")
                        display_str = f"{clar}  [{tone}]"
                        display_choices.append(display_str)
                        self.suggestion_cache[display_str] = {"clar": clar, "tone": tone, "context": ctx, "pragmatics": prag}
                    if display_choices:
                        first = self.suggestion_cache[display_choices[0]]
                        return gr.update(choices=display_choices, value=display_choices[0]), first["tone"], first["context"], first["pragmatics"], self.get_quota_status()
                    return gr.update(choices=[]), "", "", "", self.get_quota_status()
                except: return gr.update(choices=["Error"]), "Error", "", "", self.get_quota_status()

            def on_suggestion_select(val):
                if val in self.suggestion_cache:
                    return self.suggestion_cache[val]["tone"], self.suggestion_cache[val]["context"], self.suggestion_cache[val]["pragmatics"]
                return "Custom", "User provided", ""

            dialect_dropdown.change(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            orig_text_state.blur(fn=update_suggestions_rich, inputs=[orig_text_state, dialect_dropdown], outputs=[suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state, quota_display])
            suggestion_dropdown.change(fn=on_suggestion_select, inputs=[suggestion_dropdown], outputs=[selected_tone_state, selected_context_state, selected_pragmatics_state])

            audio_input.change(self.automated_pipeline, [audio_input, lang_select], [results_1, results_2, results_3, status_box, quota_display])
            analyze_btn.click(self.automated_pipeline, [audio_input, lang_select], [results_1, results_2, results_3, status_box, quota_display])

            def handle_selection(evt: gr.SelectData, df):
                if df is None or len(df) == 0: return "", "", "", "", "", ""
                try:
                    row = df.iloc[evt.index[0]]
                    if row["Source"] == "---": return "", "", "", "", "", ""
                    d = row["Dialect"] if row["Dialect"] in existing_dialects else None
                    return row["Utterance"], d, row["Clarification"], row["Tone"], row["Context"], row["Pragmatic Analysis"]
                except: return "", "", "", "", "", ""

            results_1.select(handle_selection, [results_1], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])
            results_2.select(handle_selection, [results_2], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])
            results_3.select(handle_selection, [results_3], [orig_text_state, dialect_dropdown, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state])

            def run_batch_accept(df1, df2, df3):
                return self.trust.process_batch_feedback([df1, df2, df3], self.brain, self.last_audio_path)

            def check_and_submit_logic(orig, d_drop, d_new, clar_raw, tone, context, prag):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    if not final_d or not orig: return "❌ Invalid Input", gr.update(visible=False), None
                    exists = self.trust.check_if_exists(orig, final_d, self.brain, clar_raw, tone)
                    if exists: return "⚠️ Entry already exists! Click 'Confirm Overwrite' to replace it.", gr.update(visible=True), WARNING_BEEP_PATH
                    else:
                        final_clar = str(clar_raw).rsplit("[", 1)[0].strip() if "[" in str(clar_raw) else clar_raw
                        audio_ref = self.last_audio_path
                        msg = self.trust.process_feedback("Suggest Update", orig, final_d, final_clar, tone, context, self.brain, audio_ref, prag)
                        return msg, gr.update(visible=False), None
                except Exception as e: return f"❌ Error: {e}", gr.update(visible=False), None

            def force_overwrite_logic(orig, d_drop, d_new, clar_raw, tone, context, prag):
                try:
                    final_d = d_new.strip() if d_drop == "+ Add New Dialect" else d_drop
                    final_clar = str(clar_raw).rsplit("[", 1)[0].strip() if "[" in str(clar_raw) else clar_raw
                    audio_ref = self.last_audio_path
                    msg = self.trust.process_feedback("Force Overwrite", orig, final_d, final_clar, tone, context, self.brain, audio_ref, prag)
                    return msg, gr.update(visible=False), None
                except Exception as e: return f"❌ Error: {e}", gr.update(visible=True), None

            btn_suggest.click(check_and_submit_logic, [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state], [feedback_out, btn_overwrite, warning_player])
            btn_overwrite.click(force_overwrite_logic, [orig_text_state, dialect_dropdown, new_dialect_input, suggestion_dropdown, selected_tone_state, selected_context_state, selected_pragmatics_state], [feedback_out, btn_overwrite, warning_player])
            btn_accept.click(run_batch_accept, [results_1, results_2, results_3], [feedback_out])

            def on_dialect_change(val): return gr.update(visible=True) if val == "+ Add New Dialect" else gr.update(visible=False)
            dialect_dropdown.change(on_dialect_change, inputs=dialect_dropdown, outputs=new_dialect_input)

            def change_profile(val):
                content = json.dumps(self.brain.load_profile_by_name(val), indent=2)
                return content, val
            def save_and_refresh_profile(filename, content):
                msg = self.brain.save_specific_profile(filename, content)
                new_list = self.brain.get_available_profiles()
                return msg, gr.update(choices=new_list, value=filename)
            profile_selector.change(change_profile, inputs=[profile_selector], outputs=[profile_editor, profile_filename])
            save_profile_btn.click(save_and_refresh_profile, inputs=[profile_filename, profile_editor], outputs=[profile_status, profile_selector])

        ui.launch(share=True, debug=True)

# --- START SYSTEM ---
agent1 = AgentInput()
agent2 = AgentInterpretation(gemini_manager)
agent4 = AgentTrust()
agent3 = AgentUX(agent1, agent2, agent4)
agent3.launch()



In [ ]:
import time
import sys

# 1. Measure Edge Transcription (Whisper)
start_edge = time.time()
text = whisper_model.transcribe(audio_file)
edge_latency = time.time() - start_edge

# 2. Measure Payload Size
payload = {"text": text, "persona": "NgE"}
payload_size_bytes = sys.getsizeof(str(payload))

# 3. Measure Cloud LLM Inference (Gemini via DMM)
start_cloud = time.time()
if dmm_route == "Flash":
    response = call_gemini_flash(payload)
else:
    response = call_gemini_pro(payload)
cloud_latency = time.time() - start_cloud

total_latency = edge_latency + cloud_latency

In [ ]:
# DIAGNOSTIC: List my available models
from google import genai
client = genai.Client(api_key=GOOGLE_API_KEY)
print("🔍 Scanning available models...")
for m in client.models.list():
    if "generateContent" in m.supported_actions:
        print(f" - {m.name}")

## 🚀 Getting Started with Hardhat

Hardhat is a development environment for compiling, deploying, testing, and debugging your Ethereum software. It helps developers manage and automate the recurring tasks that are inherent to building smart contracts and dApps.

### 1. Install Node.js and npm (if you don't have them)
Hardhat projects are typically set up using Node.js and its package manager, `npm`. You can download Node.js (which includes npm) from the official website: [nodejs.org](https://nodejs.org/en/download/).

### 2. Create a New Project Directory
It's best to create a dedicated directory for your Hardhat project.


In [ ]:
import os

project_name = "my-hardhat-project"
if not os.path.exists(project_name):
    os.makedirs(project_name)
    print(f"Created directory: {project_name}")
else:
    print(f"Directory '{project_name}' already exists.")

# Change to the new directory
%cd {project_name}

### 3. Initialize the Project and Install Hardhat

Inside your project directory, you'll initialize a new npm project and then install Hardhat locally.


In [ ]:
!npm init -y
!npm install --save-dev hardhat

### 4. Create a Hardhat Project

Now you can run the Hardhat command to create your first project. It will ask you to choose a project type (e.g., "Create a basic sample project"). You can select the default options.


In [ ]:
!npx hardhat

After running `npx hardhat`, you'll have a basic project structure with sample contracts, scripts, and tests. You can explore these files in the file browser (`/content/my-hardhat-project`).

### Next Steps:
*   **Explore `hardhat.config.js`**: This is where you configure your network, compilers, and plugins.
*   **Write Smart Contracts**: Look into the `contracts/` directory to start writing your Solidity code.
*   **Write Tests**: Use the `test/` directory to write tests for your contracts.
*   **Run Scripts**: The `scripts/` directory is for deployment and interaction scripts.

Let me know if you want to compile, deploy, or interact with a sample contract!

In [ ]:
import requests
import os
import json
import pandas as pd
from dotenv import load_dotenv

# Load keys
load_dotenv()
PINATA_JWT = os.getenv("PINATA_JWT")

def fetch_ipfs_logs():
    if not PINATA_JWT:
        print("❌ Error: PINATA_JWT not found.")
        return

    print("🔍 Fetching pinned files from Pinata...")

    url = "https://api.pinata.cloud/data/pinList?status=pinned"
    headers = {"Authorization": f"Bearer {PINATA_JWT}"}

    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
        files = response.json().get('rows', [])

        print(f"✅ Found {len(files)} pinned logs.")

        all_logs = []

        for file in files:
            cid = file['ipfs_pin_hash']
            # Fetch content from a public gateway
            gateway_url = f"https://gateway.pinata.cloud/ipfs/{cid}"
            try:
                log_data = requests.get(gateway_url).json()
                # Add CID for reference
                log_data['ipfs_cid'] = cid
                all_logs.append(log_data)
                print(f"   -> Retrieved log: {cid}")
            except Exception as e:
                print(f"   ⚠️ Could not read content for {cid}: {e}")

        # Convert to DataFrame for easy viewing
        if all_logs:
            df = pd.DataFrame(all_logs)
            print("\n📊 Retrieved Data Summary:")
            print(df.head())

            # Save to CSV for analysis
            df.to_csv("ipfs_audit_trail.csv", index=False)
            print("\n💾 Saved full log to 'ipfs_audit_trail.csv'")
            return df
        else:
            print("No valid logs found.")

    except Exception as e:
        print(f"❌ API Error: {e}")

# Run the retrieval
audit_df = fetch_ipfs_logs()